ФБ-31 Пилипенко Дмитро, Лаб2

In [7]:
import os
import urllib.request
import pandas as pd
import glob
from datetime import datetime
from prettytable import PrettyTable 

print("Бібліотеки імпортовано успішно")

Бібліотеки імпортовано успішно


Функція для завантаження даних

In [8]:
def download_vhi_data():
    if not os.path.exists('data'):
        os.makedirs('data')
        
    for i in range(1, 26):
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={i}&year1=1981&year2=2024&type=Mean"
        
        now = datetime.now().strftime("%Y%m%d%H%M%S")
        filename = f"data/vhi_id_{i}_{now}.csv"
        
        try:
            urllib.request.urlretrieve(url, filename)
            print(f"Завантажено ID {i} -> {filename}")
        except Exception as e:
            print(f"Помилка завантаження ID {i}: {e}")

download_vhi_data()

Завантажено ID 1 -> data/vhi_id_1_20260510121650.csv
Завантажено ID 2 -> data/vhi_id_2_20260510121651.csv
Завантажено ID 3 -> data/vhi_id_3_20260510121652.csv
Завантажено ID 4 -> data/vhi_id_4_20260510121653.csv
Завантажено ID 5 -> data/vhi_id_5_20260510121654.csv
Завантажено ID 6 -> data/vhi_id_6_20260510121655.csv
Завантажено ID 7 -> data/vhi_id_7_20260510121657.csv
Завантажено ID 8 -> data/vhi_id_8_20260510121658.csv
Завантажено ID 9 -> data/vhi_id_9_20260510121659.csv
Завантажено ID 10 -> data/vhi_id_10_20260510121700.csv
Завантажено ID 11 -> data/vhi_id_11_20260510121701.csv
Завантажено ID 12 -> data/vhi_id_12_20260510121702.csv
Завантажено ID 13 -> data/vhi_id_13_20260510121703.csv
Завантажено ID 14 -> data/vhi_id_14_20260510121704.csv
Завантажено ID 15 -> data/vhi_id_15_20260510121705.csv
Завантажено ID 16 -> data/vhi_id_16_20260510121707.csv
Завантажено ID 17 -> data/vhi_id_17_20260510121708.csv
Завантажено ID 18 -> data/vhi_id_18_20260510121709.csv
Завантажено ID 19 -> data/vh

Функція для очищення та об'єднання даних

In [9]:
def create_clean_dataframe(directory):
    files = glob.glob(os.path.join(directory, "*.csv"))
    headers = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty']
    all_data = []

    replace_map = {
        1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21,
        11: 9, 12: 13, 13: 14, 14: 15, 15: 16, 16: 25, 17: 17, 18: 18, 19: 6,
        20: 1, 21: 2, 22: 7, 23: 5, 24: 10, 25: 11
    }

    for file in files:
        orig_id = int(os.path.basename(file).split('_')[2])
        
        df = pd.read_csv(file, header=1, names=headers)
        df = df.drop('empty', axis=1) 
        
        df = df.dropna()
        df = df[df['VHI'] != -1]
        
        df['Area'] = orig_id
        df['Area'] = df['Area'].replace(replace_map)
        
        all_data.append(df)

    result_df = pd.concat(all_data, ignore_index=True)
    return result_df

df = create_clean_dataframe('data')
print("Перші 5 рядків очищеного датафрейму:")
print(df.head())

Перші 5 рядків очищеного датафрейму:
            Year  Week    SMN     SMT    VCI    TCI    VHI  Area
0  <tt><pre>1982   1.0  0.059  258.24  51.11  48.78  49.95    21
1           1982   2.0  0.063  261.53  55.89  38.20  47.04    21
2           1982   3.0  0.063  263.45  57.30  32.69  44.99    21
3           1982   4.0  0.061  265.10  53.96  28.62  41.29    21
4           1982   5.0  0.058  266.42  46.87  28.57  37.72    21


Аналіз VHI для конкретної області

In [10]:
def get_vhi_stats(dataframe, area_id, year):
    selected = dataframe[(dataframe['Area'] == area_id) & (dataframe['Year'].astype(str).str.contains(str(year)))]
    
    if selected.empty:
        return "Дані не знайдені"

    vhi_values = selected['VHI'].astype(float)
    
    table = PrettyTable()
    table.title = f"Статистика VHI для області №{area_id} за {year} рік"
    table.field_names = ["Показник", "Значення"]
    table.add_row(["Мінімум VHI", vhi_values.min()])
    table.add_row(["Максимум VHI", vhi_values.max()])
    table.add_row(["Середнє VHI", round(vhi_values.mean(), 2)])
    
    return table

print(get_vhi_stats(df, 13, 2023))

+--------------------------------------------+
| Статистика VHI для області №13 за 2023 рік |
+-------------------------+------------------+
|         Показник        |     Значення     |
+-------------------------+------------------+
|       Мінімум VHI       |      31.66       |
|       Максимум VHI      |      56.22       |
|       Середнє VHI       |      44.53       |
+-------------------------+------------------+
